In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

# 1. Load dataset
df = pd.read_csv("International_Education_Costs.csv")

# 2. Define features (X) and target (y)'' feature engineering ,selection and transformation
df['Total_Cost_USD'] = df['Tuition_USD'] + df['Rent_USD'] * df['Duration_Years'] + df['Visa_Fee_USD'] + df['Insurance_USD']
X = df.drop(columns=["Exchange_Rate",'Living_Cost_Index',"Tuition_USD", "Rent_USD", "Visa_Fee_USD", "Insurance_USD"]) #removing unwanted columns
y = df['Total_Cost_USD']   # target variable



# 3. Separate categorical and numerical columns
categorical_cols = ["Country", "City", "University", "Program", "Level"]
numerical_cols = ["Duration_Years"]

# 4. Preprocessing: OneHotEncode categorical + Scale numerical
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

#make the estimators and wrapping

estimators = [("rf",RandomForestRegressor())
              ,("br",BaggingRegressor())]
stack = StackingRegressor(estimators=estimators,final_estimator=AdaBoostRegressor())

# 5. Build pipeline with ElasticNet regression
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", stack)
])

# 6. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Fit model
model.fit(X_train, y_train)

# 8. Predictions
y_pred = model.predict(X_test)

# 9. Evaluation
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))


# 3. Apply cross-validation (cv=5)
scores = cross_val_score(model, X, y, cv=5, scoring="r2")

print("Cross-validation R² scores:", scores)
print("Average R²:", scores.mean())

# After training your pipeline model
with open("total_studycost_predictor_model.pkl", "wb") as f:
    pickle.dump(model, f)


Mean Absolute Error: 3164.8958389747027
R² Score: 0.9351477901470058
Cross-validation R² scores: [0.74177297 0.60710197 0.89358433 0.76697338 0.8930206 ]
Average R²: 0.7804906514537526
